In [1]:
import json
import time

from openai import OpenAI
import dotenv
import os
dotenv.load_dotenv()
client = OpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key=os.getenv("OPENROUTER_API_KEY"),
)

with open("dataset.txt", "r") as f:
  prompts = f.read().split("\n-\n")

In [2]:
prompts = [p.strip() for p in prompts if p.strip() != ""]
len(prompts)

228

In [3]:
import textwrap
def pprint(text):
    print(textwrap.fill(str(text), 100))

In [5]:
response = client.chat.completions.create(
    model="gpt-5.5",
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "Who is Wenqi Guo from UBC"}
            ]
        }
    ],
    extra_body={
        "reasoning": {
            "effort": "none"
        },
        "tools": [
            {"type": "openrouter:web_search"},
        ]
    } 
)

In [6]:
response.choices[0].message.content

'There appear to be **at least two UBC-linked people named Wenqi Guo**:\n\n1. **Wenqi Guo** — a former PhD student at UBC’s School of Environmental Health, now part of the School of Population and Public Health. His work focused on **air pollution exposure and cardiovascular outcomes**. He is listed by UBC’s Partnership for Work, Health and Safety as a former student. ([pwhs.ubc.ca](https://pwhs.ubc.ca/about-us/former-students/?utm_source=openai))\n\n2. **Wenqi “Marshall” Guo** — affiliated with the **University of British Columbia** in OpenReview profiles, likely in a computer science / machine learning context. Search results also show him as a UBC Okanagan mentored undergraduate thesis student in 2023. ([openreview.net](https://openreview.net/profile?id=~Wenqi_Guo3&utm_source=openai))\n\nIf you mean a specific Wenqi Guo — for example from a course, paper, lab, or UBC campus — I can help identify the right one more precisely.'

In [7]:
models = ["google/gemini-3-flash-preview",
          "openai/gpt-5.3-chat",
          "anthropic/claude-sonnet-4.6",
          "qwen/qwen3.6-plus",
          ]

In [8]:
ds = []
for prompt in prompts:
  if prompt == "":
    continue

  for model in models: 
    ds.append({
        "prompt": prompt,
        "model": model,
    })

In [9]:
from concurrent.futures import ThreadPoolExecutor, TimeoutError as FuturesTimeout

def _call_api(sample):
    return client.chat.completions.create(
        model=sample["model"],
        messages=[
            {
                "role": "system",
                "content": [
                    {"type": "text", "text": "You are a helpful assistant."}
                ]
            },
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": sample["prompt"]}
                ]
            }
        ],
        extra_body={
            "reasoning": {
                "effort": "none"
            },
            "tools": [
                {"type": "openrouter:web_search"},
            ]
        } 
    )

def eval(sample, max_retries=5):
    retries = 0
        
    while True:
        with ThreadPoolExecutor(max_workers=1) as executor:
            future = executor.submit(_call_api, sample)
            try:
                completion = future.result(timeout=60)
                if completion.choices[0].message.content:
                    sample["response"] = completion.choices[0].message.content
                    sample["model"] += ":online"
                    return sample
            except FuturesTimeout:
                retries += 1
                if retries >= max_retries:
                    raise TimeoutError("Max retries exceeded")
                continue
            except Exception as e:
                print(e)
                continue


In [10]:
len(ds)

912

In [11]:
results = {}

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed, FIRST_COMPLETED, wait
from tqdm import tqdm

executor = ThreadPoolExecutor(max_workers=30)
futures = {executor.submit(eval, ds[idx]): idx for idx in range(len(ds)) if idx not in results}

try:
    pending = set(futures)
    with tqdm(total=len(futures)) as pbar:
        while pending:
            done, pending = wait(pending, timeout=0.5, return_when=FIRST_COMPLETED)
            for future in done:
                idx = futures[future]
                results[idx] = future.result()
                pbar.update(1)
except KeyboardInterrupt:
    print("interrupted, cancelling...")
    for f in futures:
        f.cancel()
    executor.shutdown(wait=False, cancel_futures=True)
    raise
else:
    executor.shutdown()


 13%|█▎        | 117/912 [00:59<05:47,  2.29it/s]

'NoneType' object is not subscriptable


 27%|██▋       | 244/912 [01:54<04:47,  2.32it/s]

Error code: 402 - {'error': {'message': 'Insufficient credits. Add more using https://openrouter.ai/settings/credits', 'code': 402}}
Error code: 402 - {'error': {'message': 'Insufficient credits. Add more using https://openrouter.ai/settings/credits', 'code': 402}}
Error code: 402 - {'error': {'message': 'Insufficient credits. Add more using https://openrouter.ai/settings/credits', 'code': 402}}
Error code: 402 - {'error': {'message': 'Insufficient credits. Add more using https://openrouter.ai/settings/credits', 'code': 402}}
Error code: 402 - {'error': {'message': 'Insufficient credits. Add more using https://openrouter.ai/settings/credits', 'code': 402}}
Error code: 402 - {'error': {'message': 'Insufficient credits. Add more using https://openrouter.ai/settings/credits', 'code': 402}}
Error code: 402 - {'error': {'message': 'Insufficient credits. Add more using https://openrouter.ai/settings/credits', 'code': 402}}
Error code: 402 - {'error': {'message': 'Insufficient credits. Add mo

 27%|██▋       | 246/912 [01:54<03:28,  3.19it/s]

...

Error code: 402 - {'error': {'message': 'Insufficient credits. Add more using https://openrouter.ai/settings/credits', 'code': 402}}
Error code: 402 - {'error': {'message': 'Insufficient credits. Add more using https://openrouter.ai/settings/credits', 'code': 402}}
Error code: 402 - {'error': {'message': 'Insufficient credits. Add more using https://openrouter.ai/settings/credits', 'code': 402}}
Error code: 402 - {'error': {'message': 'Insufficient credits. Add more using https://openrouter.ai/settings/credits', 'code': 402}}
Error code: 402 - {'error': {'message': 'Insufficient credits. Add more using https://openrouter.ai/settings/credits', 'code': 402}}
Error code: 402 - {'error': {'message': 'Insufficient credits. Add more using https://openrouter.ai/settings/credits', 'code': 402}}
Error code: 402 - {'error': {'message': 'Insufficient credits. Add more using https://openrouter.ai/settings/credits', 'code': 402}}
Error code: 402 - {'error': {'message': 'Insufficient credits. Add mo

 30%|███       | 275/912 [02:08<04:21,  2.44it/s]

Error code: 402 - {'error': {'message': 'Insufficient credits. Add more using https://openrouter.ai/settings/credits', 'code': 402}}
Error code: 402 - {'error': {'message': 'Insufficient credits. Add more using https://openrouter.ai/settings/credits', 'code': 402}}
Error code: 402 - {'error': {'message': 'Insufficient credits. Add more using https://openrouter.ai/settings/credits', 'code': 402}}
Error code: 402 - {'error': {'message': 'Insufficient credits. Add more using https://openrouter.ai/settings/credits', 'code': 402}}
Error code: 402 - {'error': {'message': 'Insufficient credits. Add more using https://openrouter.ai/settings/credits', 'code': 402}}
Error code: 402 - {'error': {'message': 'Insufficient credits. Add more using https://openrouter.ai/settings/credits', 'code': 402}}
Error code: 402 - {'error': {'message': 'Insufficient credits. Add more using https://openrouter.ai/settings/credits', 'code': 402}}
Error code: 402 - {'error': {'message': 'Insufficient credits. Add mo

 30%|███       | 276/912 [02:08<03:48,  2.78it/s]

Error code: 402 - {'error': {'message': 'Insufficient credits. Add more using https://openrouter.ai/settings/credits', 'code': 402}}
Error code: 402 - {'error': {'message': 'Insufficient credits. Add more using https://openrouter.ai/settings/credits', 'code': 402}}
Error code: 402 - {'error': {'message': 'Insufficient credits. Add more using https://openrouter.ai/settings/credits', 'code': 402}}
Error code: 402 - {'error': {'message': 'Insufficient credits. Add more using https://openrouter.ai/settings/credits', 'code': 402}}
Error code: 402 - {'error': {'message': 'Insufficient credits. Add more using https://openrouter.ai/settings/credits', 'code': 402}}
Error code: 402 - {'error': {'message': 'Insufficient credits. Add more using https://openrouter.ai/settings/credits', 'code': 402}}
Error code: 402 - {'error': {'message': 'Insufficient credits. Add more using https://openrouter.ai/settings/credits', 'code': 402}}
Error code: 402 - {'error': {'message': 'Insufficient credits. Add mo

In [ ]:
lines = results.values()

{}

In [ ]:

with open("results.json", "r") as f:
    results_old = json.load(f)
results = results_old + list(lines)
with open("results.json", "w") as f:
    json.dump(results, f, indent=4)

0it [00:00, ?it/s]
